# Cellpose segmentation → measurement → scatter (Yokogawa)

Front-to-back basic image analysis for a Yokogawa CellVoyager run (single timepoint, no tracking):

1. **Index** the `W####F####T####Z###C#.tif` tiles in the run's `Image/` (or `Projection/`) folder
2. **Segment** objects with Cellpose (built-in Cellpose-SAM or any custom-trained model) → one label mask per tile (default: nuclei on the Hoechst channel; set `SEG_CHANNEL` to e.g. the brightfield channel for whole cells)
3. **Measure** every object in every channel (area, shape, mean/max/total intensity) + per-tile background
4. **Table**: one row per cell, columns suffixed `_C1`, `_C2`, … → CSV
5. **Scatter**: per-cell C1 vs C2, one panel per condition, coloured by point density

Run with the `cellpose` env kernel (see README). Edit only the **Config** cell; everything else is generic.
Segmentation takes ~25 s per 2000×2000 tile on Apple MPS and is cached, so re-running skips tiles that already have masks.

## Config

In [ ]:
from pathlib import Path

IMAGE_DIR = Path("/path/to/run/Image")
OUT_DIR   = Path("/path/to/run/analysis")                       # masks + CSVs + plots go here
TAG       = "my_experiment"                                                  # prefix for output files

SEG_CHANNEL   = 1                                  # channel to segment: 1 = Hoechst -> nuclei; the brightfield channel -> whole cells
CHANNEL_NAMES = {1: "Hoechst", 2: "brightfield"}   # every channel to measure -> display name (check order in the acquisition protocol)
DIAMETER      = None                               # Cellpose-SAM generally needs no diameter; set (px) if objects are mis-sized
Z_PLANE       = "auto"                     # "auto": use the projection (Z000, ZMAX, ...) if present, else the only z-plane;
                                           # or a z number (e.g. 1) to analyze one slice of a z-stack

# Cellpose model: the same choices as the GUI (run Setup to list what's installed)
#   "cpsam"                    built-in Cellpose-SAM (GUI: "run CPSAM")
#   "<name>"                   a user-trained model in ~/.cellpose/models (GUI: "custom models" dropdown)
#   "/path/to/model_file"      any model file on disk (GUI: Models menu -> "add custom model")
MODEL = "cpsam"

# Segmentation settings = GUI "additional settings" + "custom filter settings" (values below are the GUI defaults)
FLOW_THRESHOLD     = 0.4          # higher -> more (looser) masks; 0 = keep all
CELLPROB_THRESHOLD = 0.0          # lower -> more / larger masks (range about -6..6)
NORM_PERCENTILES   = (1.0, 99.0)  # intensity normalization percentiles
NITER              = 200          # flow-dynamics iterations ("niter dynamics"; GUI 0 means 200)
MIN_SIZE           = 15           # px; Cellpose drops masks smaller than this
SHARPEN_RADIUS     = 0            # image filtering: 0 = off
SMOOTH_RADIUS      = 0            #   "
TILE_NORM_BLOCKSIZE = 0           #   " (0 = normalize the whole image at once)
INVERT             = False        # invert image intensities before segmenting

# Optional subset while testing (None = everything found)
WELLS  = None        # e.g. [14, 15]   (1-based, row-major over 12 columns: B02 = 14)
FIELDS = None        # e.g. [1, 2, 3]

# Plate layout: well name -> condition label. Wells not listed are labelled by their well name.
# Double-check this against the actual plate/workup before trusting the plots.
CONDITIONS = {
    # "B02": "untreated",
    # "B03": "treated",
}

# Scatter settings
X_COL, Y_COL = "area", "mean_intensity_C1"               # any per-object column, e.g. "mean_intensity_C2", "mfi_bgsub_C2"
LOG_AXES     = True                                       # set False for bg-subtracted columns (can be <= 0)
MIN_AREA     = 50                                         # px; drop tiny debris objects

## Setup

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # torch + numpy both ship libomp on macOS

import re
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage import measure, morphology, segmentation
from tqdm import tqdm
from cellpose import models

# what the GUI offers: built-in models + user-trained models registered in ~/.cellpose/models/gui_models.txt
available = models.MODEL_NAMES + models.get_user_models()
print("Installed models:", available)

# Cellpose silently falls back to cpsam for an unknown name, so check it here
if Path(MODEL).expanduser().is_file():
    model_file = str(Path(MODEL).expanduser())
elif MODEL in available:
    model_file = MODEL
else:
    raise ValueError(f"MODEL={MODEL!r} is not an installed model or a file. Installed: {available}")
MODEL_NAME = Path(model_file).name

model = models.CellposeModel(gpu=True, pretrained_model=model_file)
print(f"Model: {MODEL_NAME} | device: {model.device}")

EVAL_KW = dict(
    diameter=DIAMETER, flow_threshold=FLOW_THRESHOLD, cellprob_threshold=CELLPROB_THRESHOLD,
    niter=NITER, min_size=MIN_SIZE,
    normalize={**models.normalize_default, "percentile": list(NORM_PERCENTILES), "sharpen_radius": SHARPEN_RADIUS,
               "smooth_radius": SMOOTH_RADIUS, "tile_norm_blocksize": TILE_NORM_BLOCKSIZE, "invert": INVERT},
)

## 1. Index tiles

In [ ]:
# W0014F0002T0001Z001C3.tif = well 14, field 2, timepoint 1, z-slice 1, channel 3.
# Projection images use Z000 (or a label such as ZMAX/ZSUM) in place of a slice number.
FNAME_RE = re.compile(r"^W(?P<well>\d+)F(?P<field>\d+)T(?P<timepoint>\d+)Z(?P<z>\d+|[A-Za-z]+)C(?P<channel>\d+)\.tiff?$",
                      re.IGNORECASE)

def well_name(w):
    """1-based row-major 96-well number -> 'B02'."""
    r, c = divmod(w - 1, 12)
    return f"{'ABCDEFGH'[r]}{c + 1:02d}"

def is_projection(z):
    return not z.isdigit() or int(z) == 0

rows = []
for p in sorted(IMAGE_DIR.iterdir()):
    m = FNAME_RE.match(p.name)
    if m:
        d = m.groupdict()
        rows.append({"path": p, "well": int(d["well"]), "field": int(d["field"]),
                     "timepoint": int(d["timepoint"]), "z": d["z"].upper(), "channel": int(d["channel"])})
if not rows:
    raise FileNotFoundError(f"No W####F####T####Z###C#.tif images in {IMAGE_DIR}")
files = pd.DataFrame(rows)
if WELLS:  files = files[files.well.isin(WELLS)]
if FIELDS: files = files[files.field.isin(FIELDS)]

# pick one z-plane: the projection if there is one, else the only slice, else what Z_PLANE says
zs = sorted(files.z.unique())
if Z_PLANE == "auto":
    proj = [z for z in zs if is_projection(z)]
    if len(proj) == 1 or len(zs) == 1:
        z_use = (proj or zs)[0]
    else:
        raise ValueError(f"Several z-planes found {zs}; set Z_PLANE to the one to analyze")
else:
    z_use = next((z for z in zs if z.isdigit() and int(z) == int(Z_PLANE)), None)
    if z_use is None:
        raise ValueError(f"Z_PLANE={Z_PLANE} not found; available: {zs}")
files = files[files.z == z_use]

# one row per tile (well/field/timepoint), one path column per channel
tiles = files.pivot_table(index=["well", "field", "timepoint", "z"], columns="channel",
                          values="path", aggfunc="first").reset_index()
needed = sorted(set(CHANNEL_NAMES) | {SEG_CHANNEL})
missing = [ch for ch in needed if ch not in tiles.columns]
if missing:
    raise ValueError(f"Channel(s) {missing} (CHANNEL_NAMES / SEG_CHANNEL) not found; available: {sorted(map(int, files.channel.unique()))}")
incomplete = tiles[needed].isna().any(axis=1)
if incomplete.any():
    print(f"Skipping {incomplete.sum()} tile(s) missing a channel image")
    tiles = tiles[~incomplete]
tiles["well_name"] = tiles.well.map(well_name)
print(f"{len(tiles)} tiles | z = Z{z_use}{' (projection)' if is_projection(z_use) else ''} | "
      f"wells {sorted(tiles.well_name.unique())} | fields {sorted(map(int, tiles.field.unique()))} | "
      f"channels {sorted(map(int, files.channel.unique()))}")

## 2. Segment — QC one tile first
Check the outlines look right before committing to the full batch (adjust `DIAMETER` if not).

In [ ]:
import json

# separate outputs per segmentation channel + model, so switching never mixes masks
RUN_DIR  = OUT_DIR / f"seg_C{SEG_CHANNEL}_{MODEL_NAME}"
MASK_DIR = RUN_DIR / "masks"
MASK_DIR.mkdir(parents=True, exist_ok=True)

# masks are cached, so record the settings that made them and stop if they have changed since
model_path = Path(model_file) if Path(model_file).is_file() else models.MODEL_DIR / model_file  # name or path -> same record
settings = {"model": str(model_path.resolve()), "seg_channel": SEG_CHANNEL, **EVAL_KW}
settings_file = RUN_DIR / "segmentation_settings.json"
if settings_file.exists() and any(MASK_DIR.iterdir()):
    old = json.loads(settings_file.read_text())
    if old != json.loads(json.dumps(settings)):
        changed = {k: (old.get(k), v) for k, v in json.loads(json.dumps(settings)).items() if old.get(k) != v}
        raise RuntimeError(f"Segmentation settings changed since the cached masks were made (old, new): {changed}\n"
                           f"Delete {MASK_DIR} to re-segment, or set a different OUT_DIR.")
settings_file.write_text(json.dumps(settings, indent=2))

def mask_path(t):
    return MASK_DIR / f"W{t.well:04d}F{t.field:04d}T{t.timepoint:04d}Z{t.z}_masks.tif"

def segment(t):
    mp = mask_path(t)
    if mp.exists():
        return tifffile.imread(mp)
    masks, _, _ = model.eval(tifffile.imread(t[SEG_CHANNEL]), **EVAL_KW)
    masks = masks.astype(np.uint16)
    tifffile.imwrite(mp, masks)
    return masks

def show_tile(t, masks, crop=800):
    img = tifffile.imread(t[SEG_CHANNEL]).astype(float)
    lo, hi = np.percentile(img, [1, 99.8])
    rgb = np.dstack([np.clip((img - lo) / (hi - lo), 0, 1)] * 3)
    rgb[segmentation.find_boundaries(masks, mode="inner")] = (1, 0.2, 0.2)
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(rgb); ax[0].set_title(f"{t.well_name} F{t.field}: {masks.max()} objects (C{SEG_CHANNEL}, {MODEL_NAME})")
    ax[1].imshow(rgb[:crop, :crop]); ax[1].set_title(f"top-left {crop}px zoom")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

qc = tiles.iloc[0]
show_tile(qc, segment(qc))

## 3. Segment all tiles + measure every channel
Per-tile background = median of pixels outside the segmented objects (masks dilated 10 px), recorded per channel; `mfi_bgsub_C*` = mean intensity − that background.

In [ ]:
PROPS = ["label", "area", "centroid", "eccentricity", "solidity", "perimeter",
         "major_axis_length", "minor_axis_length", "mean_intensity", "max_intensity"]

per_tile = []
for _, t in tqdm(tiles.iterrows(), total=len(tiles)):
    masks = segment(t)
    if masks.max() == 0:
        continue
    bg_px = ~morphology.binary_dilation(masks > 0, morphology.disk(10))
    shape = pd.DataFrame(measure.regionprops_table(masks, properties=PROPS[:8]))
    shape = shape.rename(columns={"centroid-0": "centroid_y", "centroid-1": "centroid_x"})
    for ch in CHANNEL_NAMES:
        img = tifffile.imread(t[ch])
        inten = measure.regionprops_table(masks, intensity_image=img, properties=["mean_intensity", "max_intensity"])
        bg = float(np.median(img[bg_px]))
        shape[f"mean_intensity_C{ch}"] = inten["mean_intensity"]
        shape[f"max_intensity_C{ch}"]  = inten["max_intensity"]
        shape[f"total_intensity_C{ch}"] = inten["mean_intensity"] * shape["area"]
        shape[f"bg_C{ch}"] = bg
        shape[f"mfi_bgsub_C{ch}"] = inten["mean_intensity"] - bg
    shape.insert(0, "well", t.well); shape.insert(1, "well_name", t.well_name)
    shape.insert(2, "field", t.field); shape.insert(3, "timepoint", t.timepoint); shape.insert(4, "z", t.z)
    per_tile.append(shape)

cells_df = pd.concat(per_tile, ignore_index=True)
cells_df = cells_df[cells_df.area >= MIN_AREA]
cells_df["condition"] = cells_df.well_name.map(CONDITIONS).fillna(cells_df.well_name)
cells_df.to_csv(RUN_DIR / f"{TAG}_cells.csv", index=False)
print(f"{len(cells_df)} cells -> {RUN_DIR / f'{TAG}_cells.csv'}")

## 4. Per-well summary

In [ ]:
meas = ["area"] + [f"{s}_C{ch}" for ch in CHANNEL_NAMES for s in ("mean_intensity", "mfi_bgsub", "bg")]
summary = (cells_df.groupby(["condition", "well_name"])
           .agg(n_cells=("label", "size"), **{f"median_{c}": (c, "median") for c in meas})
           .round(1).reset_index())
summary.to_csv(RUN_DIR / f"{TAG}_well_summary.csv", index=False)
summary

## 5. Scatter: per-cell C1 vs C2 by condition
Each point is one segmented object (nucleus or cell), coloured by local point density (light = sparse, dark = dense; densest drawn on top). Axes are shared across panels and clipped to the 0.5–99.5th percentile of pooled data.

In [ ]:
def label(col):
    m = re.search(r"_C(\d+)$", col)
    return col.replace(m.group(0), f" ({CHANNEL_NAMES[int(m.group(1))]})") if m else col

d = cells_df[[X_COL, Y_COL, "condition"]].dropna()
if LOG_AXES:
    d = d[(d[X_COL] > 0) & (d[Y_COL] > 0)]
tf = np.log10 if LOG_AXES else (lambda v: v)
xlim = np.percentile(tf(d[X_COL]), [0.5, 99.5])
ylim = np.percentile(tf(d[Y_COL]), [0.5, 99.5])

def density(x, y, bins=120, sigma=2):
    h, xe, ye = np.histogram2d(x, y, bins=bins, range=[xlim, ylim])
    h = gaussian_filter(h, sigma)
    ix = np.clip(np.digitize(x, xe) - 1, 0, bins - 1)
    iy = np.clip(np.digitize(y, ye) - 1, 0, bins - 1)
    return h[ix, iy]

conds = list(dict.fromkeys(list(CONDITIONS.values()) + sorted(d.condition.unique())))
conds = [c for c in conds if c in set(d.condition)]
ncol = min(4, len(conds)); nrow = int(np.ceil(len(conds) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.8 * nrow), sharex=True, sharey=True, squeeze=False)
for ax, cond in zip(axes.flat, conds):
    s = d[d.condition == cond]
    x, y = tf(s[X_COL].to_numpy()), tf(s[Y_COL].to_numpy())
    keep = (x >= xlim[0]) & (x <= xlim[1]) & (y >= ylim[0]) & (y <= ylim[1])
    x, y = x[keep], y[keep]
    z = density(x, y); o = np.argsort(z)
    ax.scatter(x[o], y[o], c=z[o], s=4, cmap="viridis", linewidths=0, rasterized=True)
    ax.set_title(f"{cond}  (n={len(s)})", fontsize=10)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
for ax in axes.flat[len(conds):]:
    ax.axis("off")
pre = "log10 " if LOG_AXES else ""
fig.supxlabel(pre + label(X_COL)); fig.supylabel(pre + label(Y_COL))
fig.suptitle(TAG, fontweight="bold")
plt.tight_layout()
fig.savefig(RUN_DIR / f"{TAG}_scatter_{X_COL}_vs_{Y_COL}.png", dpi=200)
plt.show()